# ClinicalAgent - Basic RAG Baseline

A single-pass keyword RAG baseline over synthetic clinical notes, following the course structure: a `search` function, a `build_prompt` function, an `llm` function, and a top-level `rag` function that chains them together.

The notes are rendered from a 50-patient Synthea cohort by `scripts/render_notes.py`: one note per encounter, stored as one record per section.

In [ ]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url=os.environ["GROQ_BASE_URL"],
)

## 1. Load the Clinical Notes

Each file in `data/notes/` is one encounter note, saved as a list of section records with the fields `doc_id`, `patient_id`, `document_type`, `date`, `section`, `title` and `content`.

In [ ]:
notes_dir = Path("../data/notes")

documents = []
for path in sorted(notes_dir.glob("*.json")):
    documents.extend(json.loads(path.read_text(encoding="utf-8")))

print(f"Loaded {len(documents)} section records from {len(list(notes_dir.glob('*.json')))} notes")
documents[0]

## 2. Build the Index

minsearch builds an in-memory TF-IDF index. `title`, `section` and `content` are free text to search over. `patient_id`, `doc_id`, `document_type` and `date` are keyword fields: they are not searched, but can be used to filter results exactly, e.g. `filter_dict={"patient_id": "..."}`.

In [ ]:
index = Index(
    text_fields=["title", "section", "content"],
    keyword_fields=["patient_id", "doc_id", "document_type", "date"],
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [ ]:
def search(query):
    return index.search(query, num_results=5)

In [ ]:
instructions = """
You're a clinical records assistant. Answer the QUESTION about patients'
clinical records using only the documents in the CONTEXT below.

Rules:
- Use only facts written in the CONTEXT documents. Never infer clinical facts
  that aren't written down - no assumed diagnoses, medications, doses or
  outcomes, even if they seem clinically likely.
- Cite every claim with its source as [doc_id, section], for example
  [aee7bbe1-0c45-c028-dd01-57fc9e070dc2, Medications].
- If the CONTEXT doesn't contain the answer, or only part of it, say so
  plainly and state which part is missing. Don't guess.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [ ]:
def llm(user_prompt, instructions=None, model='openai/gpt-oss-120b'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [ ]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

A multi-hop question: the diagnosis and the medication list are recorded at different encounters, so they live in different documents.

In [ ]:
query = "Which patient has neuropathy due to type 2 diabetes, and what medications are they on?"

print(rag(query))